# transdiagnostic subtyping
- cluster analysis

In [ ]:
# Transdiagnostic Computational Psychiatry Study
# High-Impact Subtype Discovery Across Neurodevelopmental and Psychiatric Conditions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import umap
import hdbscan
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("=== TRANSDIAGNOSTIC COMPUTATIONAL PSYCHIATRY STUDY ===")
print("Objective: Identify data-driven subtypes across neurodevelopmental and psychiatric conditions")

In [ ]:
# Load the raw C4 dataset for maximum sample size and features
df_raw = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/raw/data_c4_raw.csv')

print("=== DATASET EXPLORATION ===")
print(f"Original dataset shape: {df_raw.shape}")
print(f"Total participants: {len(df_raw):,}")

# Remove test entries (first 16 rows)
df_raw = df_raw.iloc[16:].reset_index(drop=True)

# Remove records with missing compulsory data
compulsory_cols = ['age', 'sex', 'handedness', 'education', 'occupation', 'country_region']
df_clean = df_raw.dropna(subset=compulsory_cols)

print(f"After cleaning: {df_clean.shape}")
print(f"Final sample size: {len(df_clean):,} participants")

# Explore diagnosis columns
diagnosis_cols = [col for col in df_clean.columns if 'diagnosis' in col]
print(f"\nDiagnosis columns: {diagnosis_cols}")

# Show diagnosis distribution
print("\nDiagnosis distribution:")
for col in diagnosis_cols[:5]:  # Show first 5
    if col in df_clean.columns:
        print(f"{col}: {df_clean[col].value_counts().head(3)}")

# create transdiagnostic features

In [ ]:
# Create transdiagnostic features for clustering
print("=== CREATING TRANSDIAGNOSTIC FEATURES ===")

# 1. Calculate questionnaire totals
questionnaire_cols = ['aq_1', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6', 'aq_7', 'aq_8', 'aq_9', 'aq_10',
                     'eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5', 'eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10',
                     'spq_1', 'spq_2', 'spq_3', 'spq_4', 'spq_5', 'spq_6', 'spq_7', 'spq_8', 'spq_9', 'spq_10',
                     'sqr_1', 'sqr_2', 'sqr_3', 'sqr_4', 'sqr_5', 'sqr_6', 'sqr_7', 'sqr_8', 'sqr_9', 'sqr_10']

# Calculate totals
df_clean['aq_total'] = df_clean[['aq_1', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6', 'aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
df_clean['eq_total'] = df_clean[['eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5', 'eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10']].sum(axis=1)
df_clean['spq_total'] = df_clean[['spq_1', 'spq_2', 'spq_3', 'spq_4', 'spq_5', 'spq_6', 'spq_7', 'spq_8', 'spq_9', 'spq_10']].sum(axis=1)
df_clean['sqr_total'] = df_clean[['sqr_1', 'sqr_2', 'sqr_3', 'sqr_4', 'sqr_5', 'sqr_6', 'sqr_7', 'sqr_8', 'sqr_9', 'sqr_10']].sum(axis=1)

# 2. Create diagnosis indicators (binary)
diagnosis_mapping = {
    'ADHD': 1, 'Autism': 2, 'Bipolar': 3, 'Depression': 4, 
    'Learning': 5, 'OCD': 6, 'Schizophrenia': 7, 'Other': 8, 'None': 9
}

# Create binary diagnosis columns
for condition, code in diagnosis_mapping.items():
    if condition != 'None':
        mask = (df_clean['diagnosis_0'] == code) | (df_clean['diagnosis_1'] == code) | \
               (df_clean['diagnosis_2'] == code) | (df_clean['diagnosis_3'] == code) | \
               (df_clean['diagnosis_4'] == code) | (df_clean['diagnosis_5'] == code) | \
               (df_clean['diagnosis_6'] == code) | (df_clean['diagnosis_7'] == code) | \
               (df_clean['diagnosis_8'] == code)
        df_clean[f'{condition.lower()}_diagnosis'] = mask.astype(int)

# 3. Create comorbidity features
diagnosis_cols_binary = [col for col in df_clean.columns if col.endswith('_diagnosis')]
df_clean['comorbidity_count'] = df_clean[diagnosis_cols_binary].sum(axis=1)
df_clean['has_multiple_diagnoses'] = (df_clean['comorbidity_count'] > 1).astype(int)

# 4. Create demographic features
df_clean['age_group'] = pd.cut(df_clean['age'], bins=[0, 18, 25, 35, 50, 100], labels=['<18', '18-25', '26-35', '36-50', '50+'])
df_clean['is_stem'] = (df_clean['occupation'] == 3).astype(int)  # STEM occupation
df_clean['is_creative'] = (df_clean['occupation'] == 4).astype(int)  # Creative occupation
df_clean['is_care'] = (df_clean['occupation'] == 5).astype(int)  # Care occupation

print(f"Created {len(diagnosis_cols_binary)} diagnosis features")
print(f"Comorbidity count range: {df_clean['comorbidity_count'].min()} to {df_clean['comorbidity_count'].max()}")
print(f"Participants with multiple diagnoses: {df_clean['has_multiple_diagnoses'].sum():,} ({df_clean['has_multiple_diagnoses'].mean():.1%})")

# prepare features for clustering 

In [ ]:
# Prepare features for transdiagnostic clustering
print("=== PREPARING FEATURES FOR CLUSTERING ===")

# Select features for clustering
clustering_features = [
    # Questionnaire totals
    'aq_total', 'eq_total', 'spq_total', 'sqr_total',
    
    # Diagnosis features
    'adhd_diagnosis', 'autism_diagnosis', 'bipolar_diagnosis', 'depression_diagnosis',
    'learning_diagnosis', 'ocd_diagnosis', 'schizophrenia_diagnosis',
    'comorbidity_count', 'has_multiple_diagnoses',
    
    # Demographics
    'age', 'sex', 'education', 'occupation', 'country_region',
    'is_stem', 'is_creative', 'is_care'
]

# Create feature matrix
X_clustering = df_clean[clustering_features].copy()

# Handle missing values
X_clustering = X_clustering.fillna(X_clustering.mean())

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Features used: {clustering_features}")
print(f"Sample size for clustering: {len(X_scaled):,}")

# Show feature statistics
print("\nFeature statistics:")
print(X_clustering.describe().round(2))

# 5. dimensionality reduction for vis

In [ ]:
# Dimensionality reduction for visualization and clustering
print("=== DIMENSIONALITY REDUCTION (OPTIMIZED) ===")

# Use a subset for faster processing
sample_size = 100000  # 100K samples is still very robust
indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_scaled_subset = X_scaled[indices]
df_clean_subset = df_clean.iloc[indices].reset_index(drop=True)

print(f"Using {sample_size:,} samples for dimensionality reduction")

# 1. PCA for initial exploration
print("1. Performing PCA...")
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X_scaled_subset)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_[:5]}")
print(f"Cumulative explained variance: {np.cumsum(pca.explained_variance_ratio_[:5])}")

# 2. UMAP for visualization (optimized parameters)
print("\n2. Performing UMAP...")
umap_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.2, n_epochs=200)
X_umap = umap_reducer.fit_transform(X_scaled_subset)

# 3. Skip t-SNE (too slow)
print("3. Skipping t-SNE (too slow for large dataset)")

print("Dimensionality reduction complete!")
print(f"UMAP shape: {X_umap.shape}")
print(f"Using subset of {sample_size:,} samples")

# 5. vis data structure 

In [ ]:
# Visualize the data structure
print("=== DATA STRUCTURE VISUALIZATION ===")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. UMAP visualization
axes[0,0].scatter(X_umap[:, 0], X_umap[:, 1], alpha=0.6, s=1)
axes[0,0].set_title('UMAP Visualization')
axes[0,0].set_xlabel('UMAP 1')
axes[0,0].set_ylabel('UMAP 2')

# 2. PCA visualization (replacing t-SNE)
axes[0,1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=1)
axes[0,1].set_title('PCA Visualization (First 2 Components)')
axes[0,1].set_xlabel('PCA 1')
axes[0,1].set_ylabel('PCA 2')

# 3. Comorbidity distribution
axes[1,0].hist(df_clean_subset['comorbidity_count'], bins=range(8), alpha=0.7, edgecolor='black')
axes[1,0].set_title('Comorbidity Distribution')
axes[1,0].set_xlabel('Number of Diagnoses')
axes[1,0].set_ylabel('Count')

# 4. Age distribution
axes[1,1].hist(df_clean_subset['age'], bins=30, alpha=0.7, edgecolor='black')
axes[1,1].set_title('Age Distribution')
axes[1,1].set_xlabel('Age')
axes[1,1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("Visualization complete!")

# 7: Unsupervised Clustering - HDBSCAN

In [ ]:
# HDBSCAN clustering for transdiagnostic subtypes
print("=== HDBSCAN CLUSTERING ===")

# Use UMAP coordinates for HDBSCAN
print("1. Running HDBSCAN on UMAP coordinates...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=1000,  # Minimum cluster size
    min_samples=100,         # Minimum samples for core points
    cluster_selection_epsilon=0.1,
    cluster_selection_method='eom'
)

cluster_labels_hdbscan = clusterer.fit_predict(X_umap)

# Analyze clustering results
n_clusters_hdbscan = len(set(cluster_labels_hdbscan)) - (1 if -1 in cluster_labels_hdbscan else 0)
n_noise = list(cluster_labels_hdbscan).count(-1)

print(f"HDBSCAN Results:")
print(f"Number of clusters: {n_clusters_hdbscan}")
print(f"Noise points: {n_noise:,} ({n_noise/len(cluster_labels_hdbscan)*100:.1f}%)")

# Add cluster labels to SUBSET dataframe (not full dataframe)
df_clean_subset['hdbscan_cluster'] = cluster_labels_hdbscan

# Show cluster sizes
cluster_sizes = df_clean_subset['hdbscan_cluster'].value_counts().sort_index()
print(f"\nCluster sizes:")
for cluster_id, size in cluster_sizes.items():
    if cluster_id == -1:
        print(f"Noise: {size:,} ({size/len(df_clean_subset)*100:.1f}%)")
    else:
        print(f"Cluster {cluster_id}: {size:,} ({size/len(df_clean_subset)*100:.1f}%)")

# 8: Gaussian Mixture Model Clustering

In [ ]:
# Gaussian Mixture Model clustering (OPTIMIZED)
print("=== GAUSSIAN MIXTURE MODEL CLUSTERING ===")

# Use a smaller range and fewer initializations for speed
print("1. Finding optimal number of clusters...")

silhouette_scores = []
bic_scores = []
n_clusters_range = range(3, 8)  # Reduced range: 3-7 clusters

for n_clusters in n_clusters_range:
    print(f"Testing {n_clusters} clusters...")
    gmm = GaussianMixture(n_components=n_clusters, random_state=42, n_init=3)  # Reduced from 10 to 3
    cluster_labels = gmm.fit_predict(X_pca[:, :5])  # Use first 5 PCA components
    
    # Calculate silhouette score
    if len(set(cluster_labels)) > 1:
        sil_score = silhouette_score(X_pca[:, :5], cluster_labels)
        silhouette_scores.append(sil_score)
    else:
        silhouette_scores.append(0)
    
    # Calculate BIC
    bic_scores.append(gmm.bic(X_pca[:, :5]))

# Find optimal number of clusters
optimal_n_clusters_sil = n_clusters_range[np.argmax(silhouette_scores)]
optimal_n_clusters_bic = n_clusters_range[np.argmin(bic_scores)]

print(f"Optimal clusters (Silhouette): {optimal_n_clusters_sil}")
print(f"Optimal clusters (BIC): {optimal_n_clusters_bic}")

# Use silhouette-based optimal number
optimal_n_clusters = optimal_n_clusters_sil
print(f"Using {optimal_n_clusters} clusters based on silhouette score")

# Fit final GMM with more initializations for final model
print("2. Fitting final GMM model...")
gmm_final = GaussianMixture(n_components=optimal_n_clusters, random_state=42, n_init=5)
cluster_labels_gmm = gmm_final.fit_predict(X_pca[:, :5])

# Add cluster labels to SUBSET dataframe
df_clean_subset['gmm_cluster'] = cluster_labels_gmm

# Show cluster sizes
cluster_sizes_gmm = df_clean_subset['gmm_cluster'].value_counts().sort_index()
print(f"\nGMM Cluster sizes:")
for cluster_id, size in cluster_sizes_gmm.items():
    print(f"Cluster {cluster_id}: {size:,} ({size/len(df_clean_subset)*100:.1f}%)")

# 9. vis clustering results

In [ ]:
# Visualize clustering results
print("=== CLUSTERING VISUALIZATION ===")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. HDBSCAN clusters on UMAP
scatter1 = axes[0,0].scatter(X_umap[:, 0], X_umap[:, 1], c=cluster_labels_hdbscan, 
                             cmap='tab10', alpha=0.6, s=1)
axes[0,0].set_title('HDBSCAN Clusters (UMAP)')
axes[0,0].set_xlabel('UMAP 1')
axes[0,0].set_ylabel('UMAP 2')

# 2. GMM clusters on UMAP
scatter2 = axes[0,1].scatter(X_umap[:, 0], X_umap[:, 1], c=cluster_labels_gmm, 
                             cmap='tab10', alpha=0.6, s=1)
axes[0,1].set_title('GMM Clusters (UMAP)')
axes[0,1].set_xlabel('UMAP 1')
axes[0,1].set_ylabel('UMAP 2')

# 3. Cluster comparison
cluster_comparison = pd.crosstab(df_clean_subset['hdbscan_cluster'], df_clean_subset['gmm_cluster'])
axes[1,0].imshow(cluster_comparison, cmap='Blues', aspect='auto')
axes[1,0].set_title('HDBSCAN vs GMM Cluster Comparison')
axes[1,0].set_xlabel('GMM Cluster')
axes[1,0].set_ylabel('HDBSCAN Cluster')

# 4. Silhouette scores
axes[1,1].plot(n_clusters_range, silhouette_scores, 'bo-')
axes[1,1].axvline(x=optimal_n_clusters, color='red', linestyle='--', label=f'Optimal: {optimal_n_clusters}')
axes[1,1].set_title('Silhouette Score vs Number of Clusters')
axes[1,1].set_xlabel('Number of Clusters')
axes[1,1].set_ylabel('Silhouette Score')
axes[1,1].legend()

plt.tight_layout()
plt.show()

print("Clustering visualization complete!")

# 10: Characterize Clusters

In [ ]:
# Characterize clusters by their features
print("=== CLUSTER CHARACTERIZATION ===")

# Choose the clustering method to analyze (HDBSCAN or GMM)
clustering_method = 'gmm_cluster'  # Change to 'hdbscan_cluster' if preferred

# Get cluster statistics
cluster_stats = df_clean_subset.groupby(clustering_method).agg({
    # Demographics
    'age': ['mean', 'std'],
    'sex': 'mean',
    'education': 'mean',
    'occupation': 'mean',
    
    # Questionnaire scores
    'aq_total': ['mean', 'std'],
    'eq_total': ['mean', 'std'],
    'spq_total': ['mean', 'std'],
    'sqr_total': ['mean', 'std'],
    
    # Diagnoses
    'adhd_diagnosis': 'mean',
    'autism_diagnosis': 'mean',
    'bipolar_diagnosis': 'mean',
    'depression_diagnosis': 'mean',
    'learning_diagnosis': 'mean',
    'ocd_diagnosis': 'mean',
    'schizophrenia_diagnosis': 'mean',
    'comorbidity_count': ['mean', 'std'],
    'has_multiple_diagnoses': 'mean',
    
    # Occupation
    'is_stem': 'mean',
    'is_creative': 'mean',
    'is_care': 'mean'
}).round(3)

print("Cluster Characteristics:")
print(cluster_stats)

# Create cluster profiles
print("\n=== CLUSTER PROFILES ===")
n_clusters = len(cluster_stats)

for cluster_id in range(n_clusters):
    print(f"\n--- CLUSTER {cluster_id} PROFILE ---")
    
    # Get cluster data
    cluster_data = df_clean_subset[df_clean_subset[clustering_method] == cluster_id]
    
    # Demographics
    print(f"Size: {len(cluster_data):,} ({len(cluster_data)/len(df_clean_subset)*100:.1f}%)")
    print(f"Age: {cluster_data['age'].mean():.1f} ± {cluster_data['age'].std():.1f}")
    print(f"Sex (2=female): {cluster_data['sex'].mean():.3f}")
    
    # Questionnaire profiles
    print(f"AQ Total: {cluster_data['aq_total'].mean():.1f} ± {cluster_data['aq_total'].std():.1f}")
    print(f"EQ Total: {cluster_data['eq_total'].mean():.1f} ± {cluster_data['eq_total'].std():.1f}")
    print(f"SPQ Total: {cluster_data['spq_total'].mean():.1f} ± {cluster_data['spq_total'].std():.1f}")
    print(f"SQR Total: {cluster_data['sqr_total'].mean():.1f} ± {cluster_data['sqr_total'].std():.1f}")
    
    # Diagnosis prevalence
    print("Diagnosis Prevalence:")
    for diagnosis in ['adhd_diagnosis', 'autism_diagnosis', 'bipolar_diagnosis', 
                     'depression_diagnosis', 'learning_diagnosis', 'ocd_diagnosis', 'schizophrenia_diagnosis']:
        prev = cluster_data[diagnosis].mean()
        if prev > 0.05:  # Only show if >5% prevalence
            print(f"  {diagnosis.replace('_diagnosis', '').title()}: {prev:.1%}")
    
    print(f"Comorbidity: {cluster_data['comorbidity_count'].mean():.1f} ± {cluster_data['comorbidity_count'].std():.1f}")
    print(f"Multiple diagnoses: {cluster_data['has_multiple_diagnoses'].mean():.1%}")